# ORB / IB stats, conditioned on the 25-day band signal

Regime is dropped. The conditioner is now a **daily band signal**:

- `avg_range_25` = mean of daily (H−L) over the last 25 sessions
- `high_25` = highest daily High over the last 25 sessions
- **band = high_25 − 2.25 × avg_range_25**
- `clp` = (C−L)/(H−L) — where the session closed inside its own range
- **SIGNAL** = Close < band **and** clp < 0.6  (a weak close, well below the recent high)

Each session is bucketed by **how it closed relative to its band**:

| bucket | rule |
|---|---|
| **above band** | Close ≥ band |
| **under / strong close** | Close < band and clp ≥ 0.6 |
| **under / weak close (SIGNAL)** | Close < band and clp < 0.6 |

Everything downstream conditions on the **prior session's** bucket → today's outcome
(so it's a genuine next-day prediction, no lookahead). Daily bars use the `+6h`
session. Mean next-day return is unreliable on a continuous contract (roll gaps),
so direction (bull%) and **median** return are reported, never the mean.

In [ ]:
# ============================ CONFIG ============================
import numpy as np, pandas as pd
from IPython.display import display, Markdown

DATA_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
SESSION_SHIFT_HOURS = 6

BAND_LOOKBACK = 25       # sessions
BAND_MULT     = 2.25     # band = high_25 - BAND_MULT * avg_range_25
CLP_THRESH    = 0.60     # weak-close threshold on (C-L)/(H-L)
MIN_N         = 100

IB_START, IB_END, RTH_END = 570, 630, 960          # 09:30, 10:30, 16:00
ORB_ENDS = {'ORB-5': 575, 'ORB-15': 585, 'ORB-30': 600}
_ORB_END, _IB_END, _RTH_CLOSE, _FULL = 585, 630, 960, 390
print("config loaded")

In [ ]:
# ==================== LOAD + SESSION DAILY BARS + BAND ====================
_df = pd.read_csv(DATA_PATH)
_df['dt'] = pd.to_datetime(_df['Date'] + ' ' + _df['Time'], format='%Y-%m-%d %H:%M:%S')
_df = _df.set_index('dt').sort_index()
_df['sess'] = (_df.index + pd.Timedelta(hours=SESSION_SHIFT_HOURS)).normalize()
_df['tod']  = _df.index.hour*60 + _df.index.minute
print(f"{len(_df):,} 1-min rows | {_df.index[0]} -> {_df.index[-1]}")

g = _df.groupby('sess')
D = pd.DataFrame({'O':g['Open'].first(),'H':g['High'].max(),'L':g['Low'].min(),'C':g['Close'].last()})
rng = D['H'] - D['L']
D['avg_range_25'] = rng.rolling(BAND_LOOKBACK, min_periods=BAND_LOOKBACK).mean()
D['high_25']      = D['H'].rolling(BAND_LOOKBACK, min_periods=BAND_LOOKBACK).max()
D['band']         = D['high_25'] - BAND_MULT * D['avg_range_25']
D['clp']          = (D['C'] - D['L']) / (D['H'] - D['L'])
D['signal']       = (D['C'] < D['band']) & (D['clp'] < CLP_THRESH)
D['bull']         = D['C'] > D['O']                          # session candle direction
D['ret_cc']       = (D['C'] / D['C'].shift(1) - 1) * 100     # close-to-close %, this session

def _bucket(row):
    if not np.isfinite(row['band']): return np.nan
    if row['C'] >= row['band']:      return 'above band'
    if row['clp'] >= CLP_THRESH:     return 'under / strong close'
    return 'under / weak close (SIGNAL)'
D['bucket'] = D.apply(_bucket, axis=1)
D['prev_band'] = D['bucket'].shift(1)         # yesterday's bucket -> today's outcome
BAND = D[['bucket','prev_band','signal','bull','ret_cc','clp','band']]
valid = D['band'].notna()
print(f"valid sessions (post-warmup): {int(valid.sum())}")
print(f"SIGNAL fires on {int(D.loc[valid,'signal'].sum())} days = {D.loc[valid,'signal'].mean()*100:.2f}%")

In [ ]:
# ==================== BUCKET / REPORT HELPERS ====================
BUCKET_ORDER = ['above band', 'under / strong close', 'under / weak close (SIGNAL)']
PCT='{:.1f}%'; NUM='{:.1f}'; P3='{:.3f}%'; P2='{:.2f}'
def rate(col): return lambda d: d[col].mean()*100 if len(d) else None
def med(col):  return lambda d: d[col].median() if len(d) else None
def q(col,p):  return lambda d: d[col].quantile(p) if len(d) else None

def report_band(ev, metrics, title, min_n=MIN_N):
    """ev indexed by session date, must carry 'prev_band'. Bins today's outcome
    by the PRIOR session's band bucket."""
    display(Markdown(f"#### {title}"))
    v = ev[ev['prev_band'].notna()]
    def line(name, sub, gate=True):
        n=len(sub); cells=[f"{name:<28}", f"n={n:>5}"]
        if gate and n<min_n: cells.append("(below min sample)")
        else:
            for lab,fn,fmt in metrics:
                val = fn(sub) if n else None
                cells.append(f"{lab}=" + (fmt.format(val) if val is not None else "  --"))
        print("  ".join(cells))
    for b in BUCKET_ORDER: line(b, v[v['prev_band']==b])
    line('ALL', v, gate=False)
    print()
print("helpers ready")

In [ ]:
# ==================== STUDY A: signal frequency + next-day direction ====================
display(Markdown("### Band signal — frequency and next-day direction"))
vd = D[D['band'].notna()]
print(f"days analysed: {len(vd)}")
print(f"SIGNAL frequency: {vd['signal'].mean()*100:.2f}%   "
      f"(bucket mix: " + ", ".join(f"{b.split(' (')[0]} {int((vd['bucket']==b).sum())}" for b in BUCKET_ORDER) + ")")
print(f"\nBASE next-day bull%: {vd['bull'].mean()*100:.1f}   median nd ret%: {vd['ret_cc'].median():+.3f}")
sig_next = D['bull'].where(D['signal'].shift(1).fillna(False))    # day after a signal
sig_ret  = D['ret_cc'].where(D['signal'].shift(1).fillna(False))
print(f"AFTER SIGNAL next-day bull%: {sig_next.mean()*100:.1f}   median nd ret%: {sig_ret.median():+.3f}  "
      f"(n={int(D['signal'].shift(1).fillna(False).sum())})")

# next-day direction conditioned on prior session's bucket
report_band(D.assign(prev_band=D['prev_band']),
            [('bull%',rate('bull'),PCT), ('ndret_med%',med('ret_cc'),P3),
             ('clp_med',med('clp'),P2)],
            "Next-day direction by how the PRIOR session closed vs its band")

In [ ]:
# ==================== BUILD IB EVENT TABLE + attach prev_band ====================
rth = _df[(_df['tod'] >= IB_START) & (_df['tod'] < RTH_END)]
rows=[]
for sess, day in rth.groupby('sess'):
    ib=day[day['tod']<IB_END]; post=day[day['tod']>=IB_END]
    if len(ib)<30 or post.empty: continue
    o,cl=ib['Open'].iloc[0],ib['Close'].iloc[-1]; hi,lo=ib['High'].max(),ib['Low'].min()
    rng=hi-lo
    if rng<=0 or np.isnan([o,cl,hi,lo]).any(): continue
    mid=(hi+lo)/2; hi_t=ib['High'].idxmax(); lo_t=ib['Low'].idxmin()
    H=post['High'].to_numpy(); Lw=post['Low'].to_numpy(); idx=post.index
    hb=H>hi; lb=Lw<lo
    ti_hi=np.argmax(hb) if hb.any() else None; ti_lo=np.argmax(lb) if lb.any() else None
    def mins(ti): return (idx[ti]-idx[0]).total_seconds()/60+1 if ti is not None else np.nan
    hmfe=hmae=lmfe=lmae=np.nan
    if ti_hi is not None: hmfe=(H[ti_hi:].max()-hi)/hi*100; hmae=(hi-Lw[ti_hi:].min())/hi*100
    if ti_lo is not None: lmfe=(lo-Lw[ti_lo:].min())/lo*100; lmae=(H[ti_lo:].max()-lo)/lo*100
    thr=mid*0.001; C=post['Close'].to_numpy(); left=np.where(np.abs(C-mid)>thr)[0]
    mret=False; mret_t=np.nan
    if len(left):
        j0=left[0]; near=(np.abs(Lw[j0:]-mid)<=thr)|(np.abs(H[j0:]-mid)<=thr)
        if near.any(): mret=True; mret_t=(idx[j0+np.argmax(near)]-idx[j0]).total_seconds()/60
    rows.append(dict(sess=sess, ib_dir='bull' if cl>o else 'bear',
        close_half='upper' if cl>mid else 'lower', formed_first='high' if hi_t<=lo_t else 'low',
        ib_range_pct=rng/o*100, high_broke=hb.any(), low_broke=lb.any(),
        both=hb.any() and lb.any(), neither=(not hb.any()) and (not lb.any()),
        first_high=(ti_hi is not None and (ti_lo is None or ti_hi<=ti_lo)),
        t_high=mins(ti_hi), high_mfe=hmfe, high_mae=hmae, low_mfe=lmfe, low_mae=lmae,
        mid_retest=mret, mid_retest_t=mret_t))
IB = pd.DataFrame(rows).set_index('sess').join(BAND[['prev_band']])
print(f"IB days: {len(IB)} | with prev_band: {int(IB['prev_band'].notna().sum())}")

In [ ]:
# ==================== IB STATS by prior-session band bucket ====================
report_band(IB, [('high',rate('high_broke'),PCT),('low',rate('low_broke'),PCT),
                 ('both',rate('both'),PCT),('neither',rate('neither'),PCT)],
            "IB break rates")
report_band(IB, [('firstHigh',rate('first_high'),PCT),('t_hi_med',med('t_high'),NUM)],
            "IB first-break side + median minutes to break")
report_band(IB, [('bull%',(lambda d:(d['ib_dir']=='bull').mean()*100),PCT),
                 ('highFirst%',(lambda d:(d['formed_first']=='high').mean()*100),PCT)],
            "IB direction and extreme formed first")
report_band(IB, [('Hbrk_MFE',med('high_mfe'),P3),('Hbrk_MAE',med('high_mae'),P3),
                 ('Lbrk_MFE',med('low_mfe'),P3),('Lbrk_MAE',med('low_mae'),P3)],
            "IB post-break excursion, median %")
report_band(IB, [('retest%',rate('mid_retest'),PCT),('t_med',med('mid_retest_t'),NUM)],
            "IB midpoint retest rate + median minutes")

In [ ]:
# ==================== BUILD ORB TABLES + attach prev_band ====================
def build_orb(end_tod):
    seg=_df[(_df['tod']>=IB_START)&(_df['tod']<RTH_END)]; rows=[]
    for sess,day in seg.groupby('sess'):
        orb=day[day['tod']<end_tod]; rest=day[day['tod']>=end_tod]
        if len(orb)<3 or rest.empty: continue
        o,cl=orb['Open'].iloc[0],orb['Close'].iloc[-1]; hi,lo=orb['High'].max(),orb['Low'].min()
        if hi<=lo: continue
        ny_open=day['Open'].iloc[0]; ny_close=day['Close'].iloc[-1]
        ny_dir='bull' if ny_close>ny_open else 'bear'
        H=rest['High'].to_numpy(); Lw=rest['Low'].to_numpy(); idx=rest.index
        hb=H>hi; lb=Lw<lo; ti_hi=np.argmax(hb) if hb.any() else None; ti_lo=np.argmax(lb) if lb.any() else None
        if ti_hi is None and ti_lo is None: first='none'
        elif ti_lo is None or (ti_hi is not None and ti_hi<=ti_lo): first='high'
        else: first='low'
        fb_t=np.nan
        if first=='high': fb_t=(idx[ti_hi]-idx[0]).total_seconds()/60+1
        elif first=='low': fb_t=(idx[ti_lo]-idx[0]).total_seconds()/60+1
        retest=False; cont=np.nan; rmfe=rmae=np.nan
        if first in ('high','low'):
            lvl=hi if first=='high' else lo; bti=ti_hi if first=='high' else ti_lo
            aH=H[bti+1:]; aL=Lw[bti+1:]; tol=lvl*0.0002
            for k in range(len(aH)):
                touch=(abs(aL[k]-lvl)<tol) if first=='high' else (abs(aH[k]-lvl)<tol)
                inside=(aL[k]<=lvl<=aH[k])
                if touch or inside:
                    retest=True
                    if first=='high': rmfe=(aH[k:].max()-lvl)/lvl*100; rmae=(lvl-aL[k:].min())/lvl*100
                    else: rmfe=(lvl-aL[k:].min())/lvl*100; rmae=(aH[k:].max()-lvl)/lvl*100
                    nxt=slice(k+1,k+6)
                    if first=='high':
                        if aH[nxt].size and aH[nxt].max()>lvl*1.0005: cont=True
                        elif aL[nxt].size and aL[nxt].min()<lvl*0.9995: cont=False
                    else:
                        if aL[nxt].size and aL[nxt].min()<lvl*0.9995: cont=True
                        elif aH[nxt].size and aH[nxt].max()>lvl*1.0005: cont=False
                    break
        rows.append(dict(sess=sess, orb_dir='bull' if cl>o else 'bear', ny_dir=ny_dir,
            range_pct=(hi-lo)/lo*100, high_broke=hb.any(), low_broke=lb.any(),
            neither=(not hb.any()) and (not lb.any()), first_high=(first=='high'), fb_t=fb_t,
            retest=retest, cont=cont, r_mfe=rmfe, r_mae=rmae))
    d=pd.DataFrame(rows).set_index('sess').join(BAND[['prev_band']])
    d['win']=d['orb_dir']==d['ny_dir']
    return d
ORB={name:build_orb(end) for name,end in ORB_ENDS.items()}
for k,v in ORB.items(): print(f"{k}: {len(v)} days | prev_band {int(v['prev_band'].notna().sum())}")

In [ ]:
# ==================== ORB STATS by prior-session band bucket ====================
for name,d in ORB.items():
    report_band(d, [('high',rate('high_broke'),PCT),('low',rate('low_broke'),PCT),
                    ('firstHigh',rate('first_high'),PCT),('fb_med',med('fb_t'),NUM)],
                f"{name} break rates + first-break side")
    report_band(d, [('win%',rate('win'),PCT)], f"{name} direction -> NY session match")
# ORB-5 retest detail
d=ORB['ORB-5']
report_band(d, [('retest%',rate('retest'),PCT),
                ('cont%',(lambda s:s['cont'].mean()*100 if s['cont'].notna().any() else None),PCT),
                ('r_MFE',med('r_mfe'),P3),('r_MAE',med('r_mae'),P3)],
            "ORB-5 breakout-level retest + continuation + excursion")

In [ ]:
# ==================== STRUCTURAL ENGINE (survival/retrace/invalidation/post-IB) ====================
_S=_df[['Open','High','Low','Close']].rename(columns=str.lower).copy()
_S['mod']=_S.index.hour*60+_S.index.minute; _S['day']=_S.index.normalize()
def _fb(h,l,mods,hi,lo,frm):
    m=mods>=frm; up=np.flatnonzero(m&(h>hi)); dn=np.flatnonzero(m&(l<lo))
    fu=up[0] if len(up) else np.inf; fd=dn[0] if len(dn) else np.inf
    if fu==np.inf and fd==np.inf: return "none",np.nan
    if fu==fd: return "both",mods[int(fu)]
    return ("up",mods[int(fu)]) if fu<fd else ("down",mods[int(fd)])
def build_days_struct(S):
    rth=S[(S['mod']>=IB_START)&(S['mod']<_RTH_CLOSE)]; cnt=rth.groupby('day').size()
    good=set(cnt[cnt==_FULL].index); rows=[]
    for day,g in rth.groupby('day'):
        if day not in good: continue
        h=g['high'].to_numpy();l=g['low'].to_numpy();c=g['close'].to_numpy();o=g['open'].to_numpy();mods=g['mod'].to_numpy()
        io=np.flatnonzero(mods<_ORB_END); ii=np.flatnonzero(mods<_IB_END)
        ohi,olo=h[io].max(),l[io].min(); ihi,ilo=h[ii].max(),l[ii].min()
        omid=(ohi+olo)/2; imid=(ihi+ilo)/2
        o_mhi=mods[io][np.argmax(h[io])]; o_mlo=mods[io][np.argmin(l[io])]
        i_mhi=mods[ii][np.argmax(h[ii])]; i_mlo=mods[ii][np.argmin(l[ii])]
        oc=c[io][-1]; ic=c[ii][-1]; obrk,obrk_mod=_fb(h,l,mods,ohi,olo,_ORB_END); ibrk,ibrk_mod=_fb(h,l,mods,ihi,ilo,_IB_END)
        rows.append(dict(day=day, ohi=ohi,olo=olo,orng=ohi-olo,omid=omid,
            o_first='high' if o_mhi<o_mlo else ('low' if o_mlo<o_mhi else 'same'),
            o_half='upper' if oc>omid else ('lower' if oc<omid else 'mid'),
            i_first='high' if i_mhi<i_mlo else ('low' if i_mlo<i_mhi else 'same'),
            i_half='upper' if ic>imid else ('lower' if ic<imid else 'mid'),
            obrk=obrk,obrk_mod=obrk_mod,
            ihi=ihi,ilo=ilo,irng=ihi-ilo,imid=imid, ibrk=ibrk,ibrk_mod=ibrk_mod, i_mhi=i_mhi,i_mlo=i_mlo,
            orb_hi_is_ib_hi=bool(ohi==ihi), orb_lo_is_ib_lo=bool(olo==ilo)))
    Dd=pd.DataFrame(rows).set_index('day')
    Dd['surv_both']=Dd['orb_hi_is_ib_hi']&Dd['orb_lo_is_ib_lo']
    Dd['surv_neither']=~Dd['orb_hi_is_ib_hi']&~Dd['orb_lo_is_ib_lo']
    return Dd
def build_paths_struct(S,Dd):
    rth=S[(S['mod']>=IB_START)&(S['mod']<_RTH_CLOSE)]; recs=[]
    for day,g in rth.groupby('day'):
        if day not in Dd.index: continue
        r=Dd.loc[day]; h=g['high'].to_numpy();l=g['low'].to_numpy();c=g['close'].to_numpy();mods=g['mod'].to_numpy()
        rec=dict(day=day)
        if r['obrk'] in ('up','down') and r['obrk_mod']<_IB_END:
            up=r['obrk']=='up'; brk=r['ohi'] if up else r['olo']; held=r['olo'] if up else r['ohi']; after=mods>r['obrk_mod']
            def fm(mask):
                idx=np.flatnonzero(after&mask); return mods[idx[0]] if len(idx) else np.nan
            rec['mid_loss_mod']=fm(l<=r['omid'] if up else h>=r['omid']); rec['held_loss_mod']=fm(l<=held if up else h>=held)
            t_ext=r['i_mhi'] if up else r['i_mlo']
            if t_ext>r['obrk_mod']:
                w=after&(mods<=t_ext)
                if w.any():
                    deep=l[w].min() if up else h[w].max()
                    if (deep>r['ohi']) if up else (deep<r['olo']): zone='no retrace'
                    elif (deep>=r['omid']) if up else (deep<=r['omid']): zone='near half'
                    elif (deep>=r['olo']) if up else (deep<=r['ohi']): zone='far half'
                    else: zone='through held side'
                    rec['retrace_zone']=zone
        if r['ibrk'] in ('up','down'):
            up=r['ibrk']=='up'; brk=r['ihi'] if up else r['ilo']; held=r['ilo'] if up else r['ihi']; after=mods>r['ibrk_mod']
            rec['ib_cont']=(r['obrk']==r['ibrk']); rec['ib_close_beyond']=bool(c[-1]>brk) if up else bool(c[-1]<brk)
            if after.any():
                rec['ib_held_holds']=not bool((l[after]<=held).any() if up else (h[after]>=held).any())
                mfe=(h[after].max()-brk) if up else (brk-l[after].min()); rec['ib_mfe_x']=mfe/r['irng'] if r['irng']>0 else np.nan
            else: rec['ib_held_holds']=True; rec['ib_mfe_x']=0.0
        recs.append(rec)
    return pd.DataFrame(recs).set_index('day')
DAYS=build_days_struct(_S); PATHS=build_paths_struct(_S,DAYS)
DAYS=DAYS.join(BAND[['prev_band']])
print(f"structural days: {len(DAYS)} | prev_band: {int(DAYS['prev_band'].notna().sum())}")

In [ ]:
# ==================== STRUCTURAL STATS by prior-session band bucket ====================
report_band(DAYS, [('ORBhi=IBhi',rate('orb_hi_is_ib_hi'),PCT),('ORBlo=IBlo',rate('orb_lo_is_ib_lo'),PCT),
                   ('both',rate('surv_both'),PCT),('neither',rate('surv_neither'),PCT)],
            "ORB extreme survives as IB extreme")

ev=DAYS.join(PATHS[['retrace_zone']]); ev=ev[ev['retrace_zone'].notna()].copy()
for z in ['no retrace','near half','far half','through held side']: ev[f'z_{z}']=(ev['retrace_zone']==z)
report_band(ev, [('none',rate('z_no retrace'),PCT),('near',rate('z_near half'),PCT),
                 ('far',rate('z_far half'),PCT),('thruHeld',rate('z_through held side'),PCT)],
            "Retracement depth before IB extreme")

ev=DAYS.join(PATHS[['mid_loss_mod','held_loss_mod']]); ev=ev[ev['obrk'].isin(['up','down'])&(ev['obrk_mod']<_IB_END)].copy()
ev['mid_lost']=ev['mid_loss_mod'].notna(); ev['held_lost']=ev['held_loss_mod'].notna()
ev['mid_lost_min']=ev['mid_loss_mod']-ev['obrk_mod']
report_band(ev, [('midLost%',rate('mid_lost'),PCT),('heldLost%',rate('held_lost'),PCT),('t_mid_med',med('mid_lost_min'),NUM)],
            "Post-break invalidation (ORB mid / held extreme)")

ev=DAYS.join(PATHS[['ib_cont','ib_held_holds','ib_mfe_x']]); ev=ev[ev['ibrk'].isin(['up','down'])].copy()
ev['reach1x']=ev['ib_mfe_x']>=1.0
report_band(ev, [('cont%',rate('ib_cont'),PCT),('heldHolds%',rate('ib_held_holds'),PCT),
                 ('reach1xIB%',rate('reach1x'),PCT),('ext_med(xIB)',med('ib_mfe_x'),P2)],
            "Post-IB break continuation & extension (IB-range units)")

---
## Original directional / structural stats (formed-first → first break, survival, quarter)

The original tables from the structural script, computed on the same `DAYS`
table (ORB = 09:30–09:44, IB = 09:30–10:29). Each is shown **ALL** (unconditioned)
and then **by prior-session band bucket**, so you can see whether "high formed
first → breaks up" and friends shift with the band.

- s1 ORB structure → which side breaks first (formed-first / close-half / combined)
- s2 IB structure → which side breaks first
- s3 ORB break direction → IB first extreme
- survival: does the ORB extreme become the IB extreme (overall + by break direction/time)
- s4 which quarter of the hour forms the IB high / low

In [ ]:
# ==================== original directional section functions ====================
def _dir_table(frame, by, brk_col):
    """% of up/down FIRST breaks, grouped by `by` (a label Series aligned to frame)."""
    f = frame[frame[brk_col].isin(['up','down'])].copy(); f['_by'] = by.loc[f.index]
    g = f.groupby('_by')[brk_col].value_counts().unstack(fill_value=0)
    for cc in ('up','down'):
        if cc not in g: g[cc] = 0
    out = pd.DataFrame({'n': g['up']+g['down'],
                        'up_first_%': 100*g['up']/(g['up']+g['down']),
                        'down_first_%': 100*g['down']/(g['up']+g['down'])})
    nb = frame[~frame[brk_col].isin(['up','down'])]
    nb2 = nb.assign(_by=by.loc[nb.index]).groupby('_by').size()
    out['no_break'] = nb2.reindex(out.index).fillna(0).astype(int)
    return out.round(1)

def _pct(s): return round(100*float(s.mean()),1)

def s1_orb_structure(d):
    d = d[(d['o_first']!='same') & (d['o_half']!='mid')]
    return {'ORB first-extreme → break': _dir_table(d, d['o_first'], 'obrk'),
            'ORB close-half → break':   _dir_table(d, 'close '+d['o_half'], 'obrk'),
            'ORB first+half → break':   _dir_table(d, d['o_first']+' / close '+d['o_half'], 'obrk')}

def s2_ib_structure(d):
    d = d[(d['i_first']!='same') & (d['i_half']!='mid')]
    return {'IB first-extreme → break': _dir_table(d, d['i_first'], 'ibrk'),
            'IB close-half → break':    _dir_table(d, 'close '+d['i_half'], 'ibrk'),
            'IB first+half → break':    _dir_table(d, d['i_first']+' / close '+d['i_half'], 'ibrk')}

def s3_orb_to_ib(d):
    b = d[d['obrk'].isin(['up','down'])]
    rows = []
    for k, g in b.groupby('obrk'):
        rows.append(dict(cell=f"ORB broke {k}", n=len(g),
            IB_high_first_pct=_pct(g['i_first']=='high'),
            IB_low_first_pct=_pct(g['i_first']=='low'),
            IB_close_upper_pct=_pct(g['i_half']=='upper')))
    return pd.DataFrame(rows).set_index('cell')

def s3b_survival(d):
    overall = pd.DataFrame([dict(cell='all days', n=len(d),
        ORBhi_is_IBhi_pct=_pct(d['orb_hi_is_ib_hi']),
        ORBlo_is_IBlo_pct=_pct(d['orb_lo_is_ib_lo']),
        both_pct=_pct(d['surv_both']), neither_pct=_pct(d['surv_neither']))]).set_index('cell')
    b = d[d['obrk'].isin(['up','down'])].copy()
    b['held_holds'] = np.where(b['obrk']=='up', b['orb_lo_is_ib_lo'], b['orb_hi_is_ib_hi'])
    rows = []
    for k, g in b.groupby('obrk'):
        rows.append(dict(cell=f"ORB broke {k}", n=len(g),
            ORBhi_is_IBhi_pct=_pct(g['orb_hi_is_ib_hi']),
            ORBlo_is_IBlo_pct=_pct(g['orb_lo_is_ib_lo']),
            held_side_holds_pct=_pct(g['held_holds'])))
    return overall, pd.DataFrame(rows).set_index('cell')

def s4_quarter(d):
    qf = lambda mod: (np.asarray(mod) - IB_START)//15
    labs = ['Q1 09:30-44','Q2 09:45-59','Q3 10:00-14','Q4 10:15-29']
    hi = pd.Series(qf(d['i_mhi'])).value_counts(normalize=True).sort_index()
    lo = pd.Series(qf(d['i_mlo'])).value_counts(normalize=True).sort_index()
    out = pd.DataFrame({'IB_high_%':(100*hi).round(1), 'IB_low_%':(100*lo).round(1)})
    out.index = [labs[i] for i in out.index]; out.index.name='quarter'
    return out
print("directional section functions ready")

In [ ]:
# ==================== ALL (unconditioned) ====================
def show(title, obj):
    display(Markdown(f"**{title}**"))
    if isinstance(obj, dict):
        for k,v in obj.items(): print(k); print(v.to_string()); print()
    else:
        print(obj.to_string()); print()

display(Markdown("### Original directional stats — ALL days"))
for t,o in s1_orb_structure(DAYS).items(): show(t,o)
for t,o in s2_ib_structure(DAYS).items(): show(t,o)
show("ORB break direction → IB first extreme", s3_orb_to_ib(DAYS))
ov, byb = s3b_survival(DAYS)
show("ORB→IB survival (overall)", ov); show("ORB→IB survival (by break dir)", byb)
show("IB extreme by quarter", s4_quarter(DAYS))

In [ ]:
# ==================== SAME STATS by prior-session band bucket ====================
for b in BUCKET_ORDER:
    dsub = DAYS[DAYS['prev_band']==b]
    display(Markdown(f"### band bucket: {b}   (n={len(dsub)})"))
    for t,o in s1_orb_structure(dsub).items(): show(t,o)
    for t,o in s2_ib_structure(dsub).items(): show(t,o)
    show("ORB break → IB first extreme", s3_orb_to_ib(dsub))
    ov, byb = s3b_survival(dsub)
    show("survival (overall)", ov)
    show("IB extreme by quarter", s4_quarter(dsub))

---
## Band parameter sweep — best settings for a next-day directional edge

Sweeps `lookback × mult × clp_threshold`, both close-strength directions (weak /
strong) and both band sides (pullback-under / rally-over). Objective: maximise the
next-day directional edge **|bull% − 50|** in either direction. Guardrails:

- **frequency floor** (signal must fire on ≥ `MIN_FRAC` of days) — kills tiny-n mirages
- **out-of-sample split** (optimise pre-2020, verify 2020→) — kills curve-fitting
- ranked by the **worst of train/test edge**, so overfit peaks sink and robust
  plateaus rise. Direction must be consistent across train & test.

Next-day bull = next session's candle closes above its open. Everything uses only
data through the current close (no lookahead).

In [ ]:
# ==================== SWEEP CONFIG ====================
SWEEP_LOOK = [10,15,20,25,30,40,50]
SWEEP_MULT = list(np.round(np.arange(1.0,3.51,0.25),2))
SWEEP_THR  = [0.3,0.4,0.5,0.6,0.7]
SPLIT_DATE = '2020-01-01'
MIN_FRAC   = 0.03          # signal must fire on >=3% of days in a split
MIN_TEST_N = 50

_rng = D['H'] - D['L']
D['nd_bull'] = (D['C'].shift(-1) > D['O'].shift(-1)).astype(float)
_tr = D.index < SPLIT_DATE; _te = D.index >= SPLIT_DATE
base_tr = D.loc[_tr,'nd_bull'].mean()*100; base_te = D.loc[_te,'nd_bull'].mean()*100
print(f"train<{SPLIT_DATE}: {int(_tr.sum())} days (base bull {base_tr:.1f}%) | "
      f"test: {int(_te.sum())} (base {base_te:.1f}%)")

In [ ]:
# ==================== RUN SWEEP ====================
rows=[]
for L in SWEEP_LOOK:
    avg=_rng.rolling(L,min_periods=L).mean()
    hi=D['H'].rolling(L,min_periods=L).max(); lo=D['L'].rolling(L,min_periods=L).min()
    for m in SWEEP_MULT:
        band_hi=hi-m*avg; band_lo=lo+m*avg
        for t in SWEEP_THR:
            for side,cpos in [('under',D['C']<band_hi),('over',D['C']>band_lo)]:
                for cdir,cclp in [('weak',D['clp']<t),('strong',D['clp']>=t)]:
                    sig=cpos & cclp
                    rec=dict(L=L,mult=m,thr=t,side=side,cdir=cdir)
                    for lbl,mask in [('train',_tr),('test',_te)]:
                        s=sig&mask; n=int(s.sum())
                        rec[f'n_{lbl}']=n
                        rec[f'bull_{lbl}']=D.loc[s,'nd_bull'].mean()*100 if n>0 else np.nan
                    rows.append(rec)
SW=pd.DataFrame(rows)
# guardrails + robust score
SW=SW[(SW['n_train']>=MIN_FRAC*_tr.sum()) & (SW['n_test']>=MIN_TEST_N)].copy()
SW['edge_train']=(SW['bull_train']-50).abs(); SW['edge_test']=(SW['bull_test']-50).abs()
SW['consistent']=np.sign(SW['bull_train']-50)==np.sign(SW['bull_test']-50)
SW['robust_edge']=np.where(SW['consistent'], SW[['edge_train','edge_test']].min(axis=1), 0.0)
SW['dir']=np.where(SW['bull_train']<50,'BEAR','bull')
print(f"combos after guardrails: {len(SW)}   (of {len(rows)} swept)")

In [ ]:
# ==================== LEADERBOARDS ====================
def _fmt(r):
    return (f"L{int(r['L']):>2} m{r['mult']:.2f} thr{r['thr']} {r['side']:<5}{r['cdir']:<6} | "
            f"train {r['bull_train']:.1f}% (n{int(r['n_train'])})  ->  "
            f"test {r['bull_test']:.1f}% (n{int(r['n_test'])})  [{r['dir']}]")

display(Markdown("#### Top 15 by robust edge (worst of train/test, consistent direction)"))
for _,r in SW.sort_values('robust_edge',ascending=False).head(15).iterrows(): print(_fmt(r))

display(Markdown("#### Best BEARISH combos (bull<50 in-sample) — do any hold out-of-sample?"))
bear=SW[SW['bull_train']<50].sort_values('edge_train',ascending=False).head(10)
if len(bear)==0: print("  none clear the frequency floor.")
for _,r in bear.iterrows(): print(_fmt(r))
n_bear_hold=int(((SW['bull_train']<50)&(SW['bull_test']<50)&(SW['n_test']>=MIN_TEST_N)).sum())
print(f"\n  bearish & holds bearish OOS: {n_bear_hold}  "
      f"(and check whether test% is actually < ~50, not just barely)")

In [ ]:
# ==================== PLATEAU MAP (under / weak family) ====================
# TEST-set bull% over lookback x mult, at a representative clp threshold.
fam = SW[(SW['side']=='under') & (SW['cdir']=='weak') & (SW['thr']==0.5)]
pt  = fam.pivot_table(index='L', columns='mult', values='bull_test')
display(Markdown("#### Plateau — next-day bull% (TEST set), under/weak, clp<0.5"))
print("rows = lookback, cols = mult ; look for a broad high-value block, not one hot cell\n")
print(pt.round(1).to_string())
print(f"\n(test base rate = {base_te:.1f}%.  A robust region well above base = real edge.)")

# concrete recommendation: best robust combo with n_test>=100 for tradeable frequency
pick = SW[(SW['n_test']>=100)].sort_values('robust_edge',ascending=False).head(1)
if len(pick):
    r=pick.iloc[0]
    display(Markdown("#### Suggested robust setting (freq-safe, n_test≥100)"))
    print(_fmt(r))
    print(f"  edge vs base:  train {r['bull_train']-base_tr:+.1f}pp   test {r['bull_test']-base_te:+.1f}pp")